In [ ]:
Homework 1 [100 points]
=======

### Deliverables:

Submit your queries (and only those) using the `submission_template.txt` file that is posted on Canvas. Follow the instructions on the file! Upload the file at Canvas.


### Instructions / Notes:

* You **may** create new IPython notebook cells to use for e.g. testing, debugging, exploring, etc.- this is encouraged in fact!- **just make sure that your final answer for each question is _in its own cell_ and _clearly indicated_**
* When you see `In [*]:` to the left of the cell you are executing, this means that the code / query is _running_.
    * **If the cell is hanging- i.e. running for too long: To restart the SQL connection, you must restart the entire python kernel**
    * To restart kernel using the menu bar: "Kernel >> Restart >> Clear all outputs & restart"), then re-execute the sql connection cell at top
    * You will also need to restart the connection if you want to load a different version of the database file
* Remember:
    * `%sql [SQL]` is for _single line_ SQL queries
    * `%%sql [SQL]` is for _multi line_ SQL queries
* _Have fun!_

Section 1: Relational Algebra [25 points]
=======

Problem 1: Relational Algebra [25 points]
---------

Consider the following relational schema for conference publications:
*  `Article(artid, title, confid, numpages)`
*  `Conference(confid, name, year, location)`
*  `Author(artid, pid)`
*  `Person(pid, name, affiliation)`

Express the following queries in the extended Relational Algebra (you can also use the aggregation operator if necessary). To write the RA expression, use the LaTex mode that ipython notebook provides. For example:

$$\pi_{name}(\sigma_{affiliation="UW-Madison"}(Person))$$ 

### Part (a) [8 points]

Output the name of every person affiliated with `UW-Madison` who has published an article in a 2021 conference.

$$
\pi_{Person.name}\Big(
\sigma_{Person.affiliation="UW\text{-}Madison"}(Person)
\ \bowtie_{Person.pid=Author.pid}\
Author
\ \bowtie_{Author.artid=Article.artid}\
Article
\ \bowtie_{Article.confid=Conference.confid}\
\sigma_{Conference.year=2021}(Conference)
\Big)
$$


### Part (b) [9 points]

Output the names of the people who coauthored an article with `John Doe`. Be careful: a person cannot be coauthor with herself!

$$
\text{Let } P_1=\rho_{P_1}(Person),\; P_2=\rho_{P_2}(Person),\;
A_1=\rho_{A_1}(Author),\; A_2=\rho_{A_2}(Author).
$$

$$
\pi_{P_2.name}\Big(
\Big(
\pi_{A_2.artid,\;A_2.pid}\big(
(\sigma_{P_1.name="John Doe"}(P_1)\ \bowtie_{P_1.pid=A_1.pid}\ A_1)
\ \bowtie_{A_1.artid=A_2.artid}\ A_2
\big)
\Big)
\;-\;
\pi_{A_1.artid,\;A_1.pid}\big(
\sigma_{P_1.name="John Doe"}(P_1)\ \bowtie_{P_1.pid=A_1.pid}\ A_1
\big)
\ \bowtie_{A_2.pid=P_2.pid}\ P_2
\Big)
$$


### Part (c) [8 points]

Translate the following SQL query to Relational Algebra.

In [ ]:
%%sql
SELECT pid, COUNT(A.artid)
FROM Article A, Conference C, Author U
WHERE A.confid = C.confid AND C.name = "PODS" AND U.artid = A.artid
GROUP BY pid ;

$$
\gamma_{U.pid;\ \text{COUNT}(A.artid)}
\Big(
\rho_A(Article)
\ \bowtie_{A.confid=C.confid}\
\sigma_{C.name="PODS"}(\rho_C(Conference))
\ \bowtie_{U.artid=A.artid}\
\rho_U(Author)
\Big)
$$


Section 2: SQL [75 points]
=======

Run the cell below to load the database `hw1.db` (make sure the database file, `hw1.db`, is in the same directory as this IPython notebook is running in)

Some of the problems involve _changing_ this database (e.g. deleting rows)- you can always re-download `hw1.db` or make a copy if you want to start fresh!

In [3]:
%load_ext sql
%sql sqlite:///hw1.db


The sql extension is already loaded. To reload it, use:
  %reload_ext sql


'Connected: @hw1.db'

Problem 2: Linear Algebra [25 points]
------------------------

Two random 3x3 ($N=3$) matrices have been provided in tables `A` and `B`, having the following schema:
> * `i INT`:   Row index
> * `j INT`:   Column index
> * `val INT`: Cell value

**Note: all of your answers below _must_ work for any _square_ matrix sizes, i.e. any value of $N$**.

Note how the matrices are represented - why do we choose this format?  Run the following queries to see the matrices in a nice format:

In [8]:
%sql SELECT group_concat(val, " , ") AS "A" FROM A GROUP BY i;

 * sqlite:///hw1.db


DatabaseError: (sqlite3.DatabaseError) database disk image is malformed
[SQL: SELECT group_concat(val, " , ") AS "A" FROM A GROUP BY i;]
(Background on this error at: https://sqlalche.me/e/14/4xp6)

In [12]:
%sql SELECT group_concat(val, " , ") AS "B" FROM B GROUP BY i;

 * sqlite:///hw1.db


DatabaseError: (sqlite3.DatabaseError) database disk image is malformed
[SQL: SELECT group_concat(val, " , ") AS "B" FROM B GROUP BY i;]
(Background on this error at: https://sqlalche.me/e/14/4xp6)

### Part (a): Matrix addition [5 points]

The sum of a matrix $A$ (having dimensions $n\times m$) and a matrix $B$ (having dimensions $n\times m$) is the matrix $C$ (of dimension $n\times m$) having cell at row $i$ and column $j$ equal to:

$C_{ij} = A_{ij} + B_{ij}$

Write a single SQL query to get the sum of $A$ and $B$ (in the same format as $A$ and $B$):

In [13]:
%%sql
SELECT A.i, A.j, A.val + B.val AS val
FROM A
JOIN B
ON A.i = B.i AND A.j = B.j
ORDER BY A.i, A.j;

 * sqlite:///hw1.db


DatabaseError: (sqlite3.DatabaseError) database disk image is malformed
[SQL: SELECT A.i, A.j, A.val + B.val AS val
FROM A
JOIN B
ON A.i = B.i AND A.j = B.j
ORDER BY A.i, A.j;]
(Background on this error at: https://sqlalche.me/e/14/4xp6)

### Part (b): Dot product [5 points]

The _dot product_ of two vectors

$a = \begin{bmatrix}a_1 & a_2 & \dots & a_n\end{bmatrix}$

and

$b = \begin{bmatrix}b_1 & b_2 & \dots & b_n\end{bmatrix}$

is

$a\cdot b = \sum_{i=1}^n a_ib_i = a_1b_1 + a_2b_2 + \dots + a_nb_n$

Write a _single SQL query_ to take the dot product of the **second column of $A$** and the **third column of $B$.**:

In [ ]:
"""
Expected output below- don't re-evaluate this cell!

NOTE: A valid answer must work for ALL inputs of the given type,
not just this example.  I.e. do not hardcode around this answer / etc!
"""

### Part (c): Matrix multiplication [7 points]

The product of a matrix $A$ (having dimensions $n\times m$) and a matrix $B$ (having dimensions $m\times p$) is the matrix $C$ (of dimension $n\times p$) having cell at row $i$ and column $j$ equal to:

$C_{ij} = \sum_{k=1}^m A_{ik}B_{kj}$

In other words, to multiply two matrices, get each cell of the resulting matrix $C$, $C_{ij}$, by taking the _dot product_ of the $i$th row of $A$ and the $j$th column of $B$.

Write a single SQL query to get the matrix product of $A$ and $B$ (in the same format as $A$ and $B$):

In [ ]:
"""
Expected output below- don't re-evaluate this cell!

NOTE: A valid answer must work for ALL inputs of the given type,
not just this example.  I.e. do not hardcode around this answer / etc!
"""

### Part (d): Matrix power [8 points]

The power $A^n$ of a matrix $A$ is defined as the matrix product of $n$ copies of $A$. 

Write a _single SQL query_ that computes the **third power** of matrix $A$, in other words, $A^3 = A \cdot A \cdot A$:

In [ ]:
"""
Expected output below- don't re-evaluate this cell!

NOTE: A valid answer must work for ALL inputs of the given type,
not just this example.  I.e. do not hardcode around this answer / etc!
"""

Problem 3: The Sales Database [25 points]
----------------------------------------------

We've prepared and loaded a dataset related to sales data from a company. The dataset has the following schema:

> `Holidays (WeekDate, IsHoliday)`

> `Stores (Store, Type, Size)`

> `TemporalData(Store, WeekDate, Temperature, FuelPrice, CPI, UnemploymentRate)`

> `Sales (Store, Dept, WeekDate, WeeklySales)`

Before you start writing queries on the database, find the schema and the constraints (keys, foreign keys). 

### Part (a): Sales during Holidays [8 points]

Using a _single SQL query_, find the store(s) with the largest overall sales during holiday weeks. Further requirements:
* Use the `WITH` clause before the main body of the query to compute a subquery if necessary.
* Return a relation with schema `(Store, AllSales)`.

Write your query here:

In [ ]:
"""
Expected output below- don't re-evaluate this cell!

NOTE: A valid answer must work for ALL inputs of the given type,
not just this example.  I.e. do not hardcode around this answer / etc!
"""

### Part (b): When Holidays do not help Sales [9 points]

Using a _single SQL query_, compute the **number** of non-holiday weeks that had larger sales than the overall average sales during holiday weeks. Further requirements:
* Use the `WITH` clause before the main body of the query to compute a subquery if necessary.
* Return a relation with schema `(NumNonHolidays)`.

Write your query here:

In [ ]:
"""
Expected output below- don't re-evaluate this cell!

NOTE: A valid answer must work for ALL inputs of the given type,
not just this example.  I.e. do not hardcode around this answer / etc!
"""

### Part (c): Total Summer Sales [8 points]

Using a _single SQL query_, compute the total sales during summer (months 6,7,and 8) for each type of store. Further requirements:
* Return a relation with schema `(type, TotalSales)`.

*Hint:* SQLite3 does not support native operations on the DATE datatype. To create a workaround, you can use the `LIKE` predicate and the string concatenation operator (||). You can also use the substring operator that SQLite3 supports (`substr`).

Write your query here:

In [ ]:
"""
Expected output below- don't re-evaluate this cell!

NOTE: A valid answer must work for ALL inputs of the given type,
not just this example.  I.e. do not hardcode around this answer / etc!
"""

Problem 4: The Traveling SQL Server Salesman Problem [25 points]
--------------------------------------------------

SQL Server salespeople are lucky as far as traveling salespeople go- they only have to sell one or two big enterprise contracts, at one or two offices in Wisconsin, in order to make their monthly quota!

Answer the following questions using the table of streets connecting company office buildings.

**Note that for convenience all streets are included _twice_, as $A \rightarrow B$ and $B \rightarrow A$.  This should make some parts of the problem easier, but remember to take it into account!**

In [ ]:
%sql SELECT * FROM streets LIMIT 4;

### Part (a): One-hop, two-hop, three-hop... [9 points]

Our salesperson has stopped at UW-Madison, to steal some cool new RDBMS technology from CS564-ers, and now wants to go sell it to a company _within 9 miles of UW-Madison and _passing through no more than 3 distinct streets_.  Write a single query, not using `WITH` (see later on), to find all such companies.

Your query should return the schema `(company, distance)` where distance is cumulative from UW-Madison.

Write your query here:

In [ ]:
"""
Expected output below- don't re-evaluate this cell!

NOTE: A valid answer must work for ALL inputs of the given type,
not just this example.  I.e. do not hardcode around this answer / etc!
"""

### Part (b): A stop at the Farm [8 points]

Now, our salesperson is out in the field, and wants to see all routes- and their distances- which will take him/her from a company $A$ to a company $B$, with the following constraints:
* The route must pass through UW-Madison (in order to pick up new RDBMS tech to sell!)
* $A$ and $B$ must _each individually_ be within 2 hops of UW-Madison
* $A$ and $B$ must be different companies
* _The total distance must be $<= 15$_
* Do not use `WITH`
* If you return a path $A \rightarrow B$, _do not include_ $B \rightarrow A$ in your answer!

In order to make your answer a bit cleaner, you may split into two queries, one of which creates a `VIEW`.  A view is a virtual table based on the output set of a SQL query.  A view can be used just like a normal table- the only difference under the hood is that the DBMS re-evaluates the query used to generate it each time a view is queried by a user (thus the data is always up-to date!)

Here's a simple example of a view:

In [ ]:
%%sql 
DROP VIEW IF EXISTS short_streets;
CREATE VIEW short_streets AS 
SELECT A, B, d FROM streets WHERE d < 3;
SELECT * FROM short_streets LIMIT 3;

Write your query or queries here:

In [ ]:

"""
Expected output below- don't re-evaluate this cell!

NOTE: A valid answer must work for ALL inputs of the given type,
not just this example.  I.e. do not hardcode around this answer / etc!
"""

### Part (c): Finding Triangles [8 points]

Finally, our salesperson wants to find a route that goes from company $A$ to company $B$ to company $C$ and then back to company $A$ with the following constraints:
* $A$, $B$, $C$ must be different companies
* Do not use `WITH` 
* Output each such route that you find once (use the id's as a way to break ties)
* Output the distance of the route

Write your query here:

In [ ]:
"""
Expected output below- don't re-evaluate this cell!

NOTE: A valid answer must work for ALL inputs of the given type,
not just this example.  I.e. do not hardcode around this answer / etc!
"""